### 📈 Volatility Risk Premium: How Options Markets Price Fear

##### ▶️ Related Quant Guild Videos:

- [The 5 Papers That Built Modern Quant Finance](https://youtu.be/ZwS1gMGegrM)

- [I Bet You've Never Found Alpha (and I Can Prove It)](https://youtu.be/UzTJHs3-eT0)

- [Quant Ranks Retail Trading Mistakes that Blow Up Your Account](https://youtu.be/1mpNxBaBeOw)

- [Non-Stationarity and Why Market Timing Fails](https://youtu.be/7nvjrgqKjJE)

- [Quant Busts 3 Trading Myths with Math](https://youtu.be/wJfIk3VnubE)

- [How to Read Options Chains](https://youtu.be/RrRbz6oXwxE)

###### ______________________________________________________________________________________________________________________________________

##### [🚀 Master your Quantitative Skills with Quant Guild](https://quantguild.com)

##### [📚 Visit the Quant Guild Library for more Jupyter Notebooks](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [📈 Interactive Brokers for Algorithmic Trading](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

##### [👾 Join the Quant Guild Discord Server](discord.com/invite/MJ4FU2c6c3)

---

##### Visualizing Historic Realized Volatility and Implied Volatility for SPX

Realized volatility is not the same thing as implied volatility.
 
 $$
     \sigma_{\text{imp}} = \arg\min_{\sigma \geq 0} \left| C_{\text{BS}}(S, K, T, r, q, \sigma) - C_{\text{mkt}} \right|
 $$

   $$
     \sigma_{\text{real}} = \sqrt{252 \cdot \operatorname{Var}\left( \ln \left( \frac{S_t}{S_{t-1}} \right) \right)}
   $$
 

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Load data ---
path = "spx_option_implied_realized_vol_10y.csv"
df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Use the supplied 21-day realized vol series and implied vol series.
# Drop the initial rows where the 21-day realized-vol window is not available.
df = df.dropna(subset=["realized_volatility", "option_implied_volatility"]).reset_index(drop=True)

dates = df["date"]
realized_vol = df["realized_volatility"]
implied_vol = df["option_implied_volatility"]

# --- Styling ---
off_white = "#e0e0e0"
realized_color = "#00ff88"
implied_color = "#33aaff"
baseline_color = "#777777"

# --- Summary stats ---
latest_date = dates.iloc[-1]
latest_rv = realized_vol.iloc[-1]
latest_iv = implied_vol.iloc[-1]
avg_rv = realized_vol.mean()
avg_iv = implied_vol.mean()
spread_latest = latest_iv - latest_rv

# --- Figure ---
fig = go.Figure()

# Long-run average realized vol baseline
fig.add_trace(go.Scatter(
    x=[dates.iloc[0], dates.iloc[-1]],
    y=[avg_rv, avg_rv],
    mode="lines",
    line=dict(color=baseline_color, width=1, dash="dash"),
    opacity=0.65,
    name=f"Avg Realized Vol · {avg_rv:.1%}",
    hoverinfo="skip"
))

# Long-run average implied vol baseline
fig.add_trace(go.Scatter(
    x=[dates.iloc[0], dates.iloc[-1]],
    y=[avg_iv, avg_iv],
    mode="lines",
    line=dict(color=baseline_color, width=1, dash="dot"),
    opacity=0.65,
    name=f"Avg Implied Vol · {avg_iv:.1%}",
    hoverinfo="skip"
))

# Animated realized volatility
fig.add_trace(go.Scatter(
    x=[dates.iloc[0]],
    y=[realized_vol.iloc[0]],
    mode="lines",
    line=dict(color=realized_color, width=3),
    name="21D Realized Vol",
    hovertemplate="Date: %{x|%Y-%m-%d}<br>21D Realized Vol: %{y:.2%}<extra></extra>"
))

# Animated implied volatility
fig.add_trace(go.Scatter(
    x=[dates.iloc[0]],
    y=[implied_vol.iloc[0]],
    mode="lines",
    line=dict(color=implied_color, width=3),
    name="Option-Implied Vol",
    hovertemplate="Date: %{x|%Y-%m-%d}<br>Implied Vol: %{y:.2%}<extra></extra>"
))

# --- Animation frames ---
frames = []
slider_steps = []

# 5-trading-day stride keeps the HTML responsive.
# Set frame_stride = 1 for a frame on every trading day.
frame_stride = 5
frame_indices = list(range(0, len(df), frame_stride))
if frame_indices[-1] != len(df) - 1:
    frame_indices.append(len(df) - 1)

for i in frame_indices:
    frame_name = f"f{i}"
    frames.append(go.Frame(
        data=[
            go.Scatter(x=[dates.iloc[0], dates.iloc[-1]], y=[avg_rv, avg_rv]),
            go.Scatter(x=[dates.iloc[0], dates.iloc[-1]], y=[avg_iv, avg_iv]),
            go.Scatter(x=dates.iloc[:i + 1], y=realized_vol.iloc[:i + 1]),
            go.Scatter(x=dates.iloc[:i + 1], y=implied_vol.iloc[:i + 1]),
        ],
        name=frame_name
    ))
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": dates.iloc[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

y_min = min(realized_vol.min(), implied_vol.min(), avg_rv, avg_iv)
y_max = max(realized_vol.max(), implied_vol.max(), avg_rv, avg_iv)
y_pad = (y_max - y_min) * 0.08

fig.update_layout(
    title=dict(
        text=(
            "SPX 21-Day Realized Volatility vs Option-Implied Volatility"
            f"<br><sup>Latest: {latest_date:%Y-%m-%d} · "
            f"RV: {latest_rv:.1%} · IV: {latest_iv:.1%} · "
            f"IV − RV: {spread_latest:.1%}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1100,
    margin=dict(t=100, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="x unified",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 20, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(
    axis_style,
    range=[dates.iloc[0], dates.iloc[-1]],
    title_text="Date"
)

fig.update_yaxes(
    axis_style,
    range=[max(0, y_min - y_pad), y_max + y_pad],
    title_text="Annualized Volatility",
    tickformat=".0%"
)
fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Realized Volatility is a *Latent Process*


Theoretically the expectation is given by:
$$\mathbb{E}[X] = \int_{-\infty}^{\infty} x\, p(x)\, dx$$

The variance, then depends on the expectation:
 $$\mathrm{Var}(X) = \mathbb{E}\left[(X - \mathbb{E}[X])^2\right]$$

Thus, realized volatility (*historic realized volatility*) depends on variance and the mean - which are arbitrary:
$$
     \sigma_{\text{real}} = \sqrt{252 \cdot \operatorname{Var}\left( \ln \left( \frac{S_t}{S_{t-1}} \right) \right)}
$$

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# Load data
# =============================================================================

path = "spx_option_implied_realized_vol_10y.csv"
df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df = df.dropna(
    subset=["realized_volatility", "option_implied_volatility"]
).reset_index(drop=True)

dates_all = df["date"]
iv_all = df["option_implied_volatility"].astype(float)
rv21_base = df["realized_volatility"].astype(float)

# =============================================================================
# Expectation / smoothing windows
# =============================================================================
# The supplied realized_volatility is already a 21D realized-vol estimate.
# These windows do NOT recompute true realized vol from returns.
# Instead, they create different expectation processes over the already-smoothed RV21 series:
#
# expectation_window = 1   -> raw RV21(t)
# expectation_window = 5   -> trailing 5D average of RV21(t)
# expectation_window = 21  -> trailing 21D average of RV21(t)
# expectation_window = 63  -> trailing 63D average of RV21(t)
#
# Effective interpretation:
# RV21 is the base realized-vol process.
# Applying another rolling expectation window makes the target smoother/slower.

expectation_windows = [1, 5, 10, 21, 30, 42, 63, 84, 126]

for w in expectation_windows:
    col = f"rv21_expectation_{w}d"

    if w == 1:
        df[col] = rv21_base
    else:
        df[col] = (
            rv21_base
            .rolling(window=w, min_periods=w)
            .mean()
        )

# Keep only rows where the largest expectation window is available.
required_cols = ["option_implied_volatility"] + [
    f"rv21_expectation_{w}d" for w in expectation_windows
]

df = df.dropna(subset=required_cols).reset_index(drop=True)

dates_all = df["date"]
iv_all = df["option_implied_volatility"].astype(float)

# =============================================================================
# Styling
# =============================================================================

off_white = "#e0e0e0"
realized_color = "#00ff88"
implied_color = "#33aaff"
regression_color = "#ff4fd8"
baseline_color = "#777777"
current_color = "#ff3e3e"

# =============================================================================
# Helper functions
# =============================================================================

def regression_stats(xv, yv):
    xv = np.asarray(xv, dtype=float)
    yv = np.asarray(yv, dtype=float)

    if len(xv) < 3 or np.std(xv, ddof=1) == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope, intercept = np.polyfit(xv, yv, 1)
    y_hat = intercept + slope * xv

    ss_res = np.sum((yv - y_hat) ** 2)
    ss_tot = np.sum((yv - np.mean(yv)) ** 2)

    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    corr = np.corrcoef(xv, yv)[0, 1]

    return slope, intercept, r2, corr


def expectation_frame_data(w):
    rv_col = f"rv21_expectation_{w}d"

    temp = df.dropna(subset=["option_implied_volatility", rv_col]).copy()

    d = temp["date"]
    iv = temp["option_implied_volatility"].astype(float).to_numpy()
    rv_expectation = temp[rv_col].astype(float).to_numpy()

    return d, iv, rv_expectation


def expectation_stats(w):
    d, iv, rv_expectation = expectation_frame_data(w)

    slope, intercept, r2, corr = regression_stats(iv, rv_expectation)

    premium = np.mean(iv - rv_expectation)
    hit_rate = np.mean(rv_expectation > iv)

    return {
        "dates": d,
        "iv": iv,
        "rv": rv_expectation,
        "slope": slope,
        "intercept": intercept,
        "r2": r2,
        "corr": corr,
        "premium": premium,
        "hit_rate": hit_rate,
        "mean_iv": float(np.mean(iv)),
        "mean_rv": float(np.mean(rv_expectation)),
        "n": len(iv)
    }


def title_text(w, stats):
    if w == 1:
        window_label = "Raw supplied RV21(t)"
        effective_note = "base 21D realized-vol process"
    else:
        window_label = f"{w}D trailing expectation of supplied RV21(t)"
        effective_note = f"21D RV process smoothed with an additional {w}D expectation window"

    return (
        "Expectation Changes Everything: IV(t) vs Realized-Vol Expectation Process"
        f"<br><sup>{window_label} · "
        f"Mean IV − RV expectation: {stats['premium']:.2%} · "
        f"RV expectation > IV share: {stats['hit_rate']:.1%} · "
        f"ρ={stats['corr']:.2f} · R²={stats['r2']:.2f} · "
        f"{effective_note}</sup>"
    )

# =============================================================================
# Global axis bounds
# =============================================================================

all_rv_expectations = []

for w in expectation_windows:
    _, _, rv_w = expectation_frame_data(w)
    all_rv_expectations.append(rv_w)

combined_vols = np.concatenate([iv_all.to_numpy()] + all_rv_expectations)

vol_min = max(0, float(np.nanmin(combined_vols) * 0.92))
vol_max = float(np.nanmax(combined_vols) * 1.08)

line_x = np.linspace(vol_min, vol_max, 250)

# =============================================================================
# Initial expectation window
# =============================================================================

w0 = 1
stats0 = expectation_stats(w0)

slope0, intercept0 = stats0["slope"], stats0["intercept"]

reg_y0 = (
    intercept0 + slope0 * line_x
    if np.isfinite(slope0)
    else np.full_like(line_x, np.nan)
)

# =============================================================================
# Figure
# =============================================================================

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.62, 0.38],
    horizontal_spacing=0.08,
    subplot_titles=(
        "Time-series comparison at date t",
        "IV(t) vs RV expectation at date t"
    )
)

# Left panel: IV(t)
fig.add_trace(
    go.Scatter(
        x=stats0["dates"],
        y=stats0["iv"],
        mode="lines",
        line=dict(color=implied_color, width=3),
        name="IV(t)",
        hovertemplate="Date: %{x|%Y-%m-%d}<br>IV(t): %{y:.2%}<extra></extra>"
    ),
    row=1,
    col=1
)

# Left panel: RV expectation
fig.add_trace(
    go.Scatter(
        x=stats0["dates"],
        y=stats0["rv"],
        mode="lines",
        line=dict(color=realized_color, width=3),
        name=f"RV21 expectation {w0}D",
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "RV expectation: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# Mean IV baseline
fig.add_trace(
    go.Scatter(
        x=[stats0["dates"].iloc[0], stats0["dates"].iloc[-1]],
        y=[stats0["mean_iv"], stats0["mean_iv"]],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dot"),
        opacity=0.65,
        name="Mean IV",
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# Mean RV expectation baseline
fig.add_trace(
    go.Scatter(
        x=[stats0["dates"].iloc[0], stats0["dates"].iloc[-1]],
        y=[stats0["mean_rv"], stats0["mean_rv"]],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.65,
        name="Mean RV expectation",
        hoverinfo="skip"
    ),
    row=1,
    col=1
)

# Right panel: y = x
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_x,
        mode="lines",
        line=dict(color=baseline_color, width=2, dash="dash"),
        opacity=0.85,
        name="y = x",
        hovertemplate="Equal IV/RV expectation: %{x:.2%}<extra></extra>"
    ),
    row=1,
    col=2
)

# Right panel: scatter
fig.add_trace(
    go.Scatter(
        x=stats0["iv"],
        y=stats0["rv"],
        mode="markers",
        marker=dict(
            color=implied_color,
            size=7,
            opacity=0.55,
            line=dict(color="rgba(255,255,255,0.22)", width=0.5)
        ),
        customdata=np.column_stack([
            stats0["dates"].dt.strftime("%Y-%m-%d"),
            stats0["iv"],
            stats0["rv"],
            stats0["iv"] - stats0["rv"]
        ]),
        name="IV vs RV expectation",
        hovertemplate=(
            "Date: %{customdata[0]}<br>"
            "IV(t): %{customdata[1]:.2%}<br>"
            "RV expectation(t): %{customdata[2]:.2%}<br>"
            "IV − RV expectation: %{customdata[3]:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Right panel: regression line
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=reg_y0,
        mode="lines",
        line=dict(color=regression_color, width=4),
        name="OLS fit",
        hovertemplate=(
            "OLS fit<br>"
            "IV: %{x:.2%}<br>"
            "Fitted RV expectation: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Right panel: mean point
fig.add_trace(
    go.Scatter(
        x=[stats0["mean_iv"]],
        y=[stats0["mean_rv"]],
        mode="markers",
        marker=dict(
            color=current_color,
            size=15,
            opacity=1,
            symbol="diamond",
            line=dict(color="rgba(255,255,255,0.95)", width=1)
        ),
        name="Mean point",
        hovertemplate=(
            "Mean IV: %{x:.2%}<br>"
            "Mean RV expectation: %{y:.2%}<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# =============================================================================
# Animation frames
# =============================================================================

frames = []
slider_steps = []

for w in expectation_windows:
    stats = expectation_stats(w)

    slope, intercept = stats["slope"], stats["intercept"]

    reg_y = (
        intercept + slope * line_x
        if np.isfinite(slope)
        else np.full_like(line_x, np.nan)
    )

    frame_name = f"w{w}"

    frames.append(go.Frame(
        data=[
            # 0 IV(t)
            go.Scatter(
                x=stats["dates"],
                y=stats["iv"]
            ),

            # 1 RV expectation
            go.Scatter(
                x=stats["dates"],
                y=stats["rv"],
                name=f"RV21 expectation {w}D"
            ),

            # 2 Mean IV
            go.Scatter(
                x=[stats["dates"].iloc[0], stats["dates"].iloc[-1]],
                y=[stats["mean_iv"], stats["mean_iv"]]
            ),

            # 3 Mean RV expectation
            go.Scatter(
                x=[stats["dates"].iloc[0], stats["dates"].iloc[-1]],
                y=[stats["mean_rv"], stats["mean_rv"]]
            ),

            # 5 Scatter
            go.Scatter(
                x=stats["iv"],
                y=stats["rv"],
                customdata=np.column_stack([
                    stats["dates"].dt.strftime("%Y-%m-%d"),
                    stats["iv"],
                    stats["rv"],
                    stats["iv"] - stats["rv"]
                ])
            ),

            # 6 Regression
            go.Scatter(
                x=line_x,
                y=reg_y
            ),

            # 7 Mean point
            go.Scatter(
                x=[stats["mean_iv"]],
                y=[stats["mean_rv"]]
            )
        ],
        traces=[0, 1, 2, 3, 5, 6, 7],
        layout=go.Layout(
            title=dict(
                text=title_text(w, stats),
                x=0.5,
                font=dict(color=off_white)
            )
        ),
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": f"{w}D",
        "method": "animate"
    })

fig.frames = frames

# =============================================================================
# Layout
# =============================================================================

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

fig.update_layout(
    title=dict(
        text=title_text(w0, stats0),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=720,
    width=1300,
    margin=dict(t=115, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.04,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 750, "redraw": False},
                        "transition": {"duration": 250},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": expectation_windows.index(w0),
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "RV expectation window: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(
    axis_style,
    range=[dates_all.iloc[0], dates_all.iloc[-1]],
    title_text="Date t",
    row=1,
    col=1
)

fig.update_yaxes(
    axis_style,
    range=[vol_min, vol_max],
    title_text="Annualized Volatility",
    tickformat=".0%",
    row=1,
    col=1
)

fig.update_xaxes(
    axis_style,
    range=[vol_min, vol_max],
    title_text="IV(t)",
    tickformat=".0%",
    row=1,
    col=2
)

fig.update_yaxes(
    axis_style,
    range=[vol_min, vol_max],
    title_text="RV21 Expectation Process at t",
    tickformat=".0%",
    scaleanchor="x2",
    scaleratio=1,
    row=1,
    col=2
)

for annotation in fig.layout.annotations:
    annotation.font = dict(color=off_white, size=15)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Implied Volatility Prices *Option Contracts*

 $$
     \sigma_{\text{imp}} = \arg\min_{\sigma \geq 0} \left| C_{\text{BS}}(S, K, T, r, q, \sigma) - C_{\text{mkt}} \right|
 $$


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Load local vol-history data for calibration ---
path = "spx_option_implied_realized_vol_10y.csv"
df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
df = df.dropna(subset=["realized_volatility", "option_implied_volatility"]).reset_index(drop=True)

iv_hist = df["option_implied_volatility"].to_numpy()
rv_hist = df["realized_volatility"].to_numpy()

latest_date = df["date"].iloc[-1]
latest_iv = float(df["option_implied_volatility"].iloc[-1])
avg_iv = float(np.mean(iv_hist))
p75_iv = float(np.percentile(iv_hist, 75))
p90_iv = float(np.percentile(iv_hist, 90))

# Current SPY reference price from finance quote pulled in the prior step.
spot = 754.83
symbol = "SPY"

# --- Build a realistic non-live IV surface ---
dte = np.array([7, 14, 21, 30, 45, 60, 90, 120, 180], dtype=float)
moneyness = np.linspace(0.82, 1.18, 45)
strikes = spot * moneyness

K, T = np.meshgrid(strikes, dte)
M = K / spot
tau = T / 365.0

# Calibrate surface
atm_anchor = max(0.08, latest_iv)
front_premium = 0.055 * np.exp(-T / 38.0)
term_slope = 0.018 * np.log1p(T / 45.0)
left_skew = 0.22 * np.maximum(1 - M, 0) ** 1.25 * np.exp(-T / 260.0)
right_wing = 0.055 * np.maximum(M - 1, 0) ** 1.55
smile = 0.30 * (np.log(M) ** 2)
ripple = 0.004 * np.sin(12 * (M - 1)) * np.exp(-T / 150.0)

Z = atm_anchor + front_premium + term_slope + left_skew + right_wing + smile + ripple

Z = np.clip(Z, max(0.05, np.percentile(iv_hist, 2) * 0.75), max(0.80, p90_iv * 2.2))

# --- Styling ---
off_white = "#e0e0e0"
paper = "rgba(0,0,0,0)"
plot_bg = "rgba(0,0,0,0)"

# --- Figure ---
fig = go.Figure()

# Only the clean implied volatility surface, no lines or labels
fig.add_trace(go.Surface(
    x=K,
    y=T,
    z=Z,
    colorscale="Magma",
    opacity=0.92,
    name="Implied Vol Surface",
    colorbar=dict(
        title=dict(text="IV", font=dict(color=off_white)),
        tickfont=dict(color=off_white),
        tickformat=".0%",
        len=0.72,
        x=0.94
    ),
    contours=dict(
        z=dict(show=False),
        x=dict(show=False),
        y=dict(show=False)
    ),
    hovertemplate=(
        "Strike: %{x:.2f}<br>"
        "DTE: %{y:.0f}<br>"
        "Implied Vol: %{z:.2%}<extra></extra>"
    )
))

z_floor = float(np.min(Z) * 0.985)

# Camera/view buttons
camera_default = dict(eye=dict(x=1.65, y=-1.85, z=1.15), center=dict(x=0, y=0, z=-0.05))
camera_top = dict(eye=dict(x=0.01, y=0.01, z=2.65), center=dict(x=0, y=0, z=0))
camera_skew = dict(eye=dict(x=0.0, y=-2.45, z=0.55), center=dict(x=0, y=0, z=-0.08))
camera_term = dict(eye=dict(x=2.45, y=0.0, z=0.55), center=dict(x=0, y=0, z=-0.08))

fig.update_layout(
    title=dict(
        text=(
            f"{symbol} Static Implied Volatility Surface"
            f"<br><sup>Transparent 3D Plotly view · Spot ref: {spot:.2f} · "
            f"Calibrated ATM IV: {latest_iv:.1%} · Latest vol-history date: {latest_date:%Y-%m-%d}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor=paper,
    plot_bgcolor=plot_bg,
    height=760,
    width=1180,
    margin=dict(t=105, b=55, r=25, l=25),
    legend=dict(
        x=0.02,
        y=0.98,
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    scene=dict(
        bgcolor="rgba(0,0,0,0)",
        xaxis=dict(
            title=dict(text="Strike", font=dict(color=off_white)),
            tickfont=dict(color=off_white),
            gridcolor="rgba(255,255,255,0.10)",
            zerolinecolor="rgba(255,255,255,0.12)",
            showbackground=False,
            range=[strikes.min(), strikes.max()]
        ),
        yaxis=dict(
            title=dict(text="Days to Expiry", font=dict(color=off_white)),
            tickfont=dict(color=off_white),
            gridcolor="rgba(255,255,255,0.10)",
            zerolinecolor="rgba(255,255,255,0.12)",
            showbackground=False,
            range=[dte.min(), dte.max()]
        ),
        zaxis=dict(
            title=dict(text="Implied Volatility", font=dict(color=off_white)),
            tickfont=dict(color=off_white),
            tickformat=".0%",
            gridcolor="rgba(255,255,255,0.10)",
            zerolinecolor="rgba(255,255,255,0.12)",
            showbackground=False,
            range=[z_floor, float(np.max(Z) * 1.05)]
        ),
        camera=camera_default,
        aspectmode="manual",
        aspectratio=dict(x=1.55, y=1.05, z=0.65)
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=0.5,
            y=0.02,
            xanchor="center",
            yanchor="bottom",
            showactive=True,
            bgcolor="rgba(30,30,30,0.8)",
            bordercolor="rgba(255,255,255,0.25)",
            borderwidth=1,
            font=dict(color=off_white),
            buttons=[
                dict(
                    label="Default 3D",
                    method="relayout",
                    args=[{"scene.camera": camera_default}]
                ),
                dict(
                    label="Top Surface",
                    method="relayout",
                    args=[{"scene.camera": camera_top}]
                ),
                dict(
                    label="Skew View",
                    method="relayout",
                    args=[{"scene.camera": camera_skew}]
                ),
                dict(
                    label="Term View",
                    method="relayout",
                    args=[{"scene.camera": camera_term}]
                )
            ]
        )
    ]
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

#### What is the Statistical Relationship Between Realized and Implied Volatility?

Regression equation (implied vs. realized volatility):
 
  $$
      (\beta_0,\,\beta_1) = \operatorname*{arg\,min}_{\beta_0,\,\beta_1}\;\sum_{t=1}^N \left( \operatorname{RV}_{t+\Delta}^2 - (\beta_0 + \beta_1 \operatorname{IV}_t^2) \right)^2
  $$


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Load data ---
path = "spx_option_implied_realized_vol_10y.csv"
df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df = df.dropna(subset=["realized_volatility", "option_implied_volatility"]).reset_index(drop=True)

# --- Forward 30-trading-day realized volatility ---
# Since realized_volatility is already a rolling realized-vol estimate,
# for date t, y = realized_volatility at t + 30 trading days.
forward_window = 30

df["forward_30d_realized_volatility"] = (
    df["realized_volatility"]
    .astype(float)
    .shift(-forward_window)
)

scatter_df = df.dropna(
    subset=["option_implied_volatility", "forward_30d_realized_volatility"]
).copy()

x = scatter_df["option_implied_volatility"].to_numpy(dtype=float)
y = scatter_df["forward_30d_realized_volatility"].to_numpy(dtype=float)
dates = scatter_df["date"].reset_index(drop=True)

# --- Styling ---
off_white = "#e0e0e0"
point_color = "#33aaff"
current_color = "#00ff88"
regression_color = "#ff4fd8"
baseline_color = "#777777"
zero_color = "#ff3e3e"

# --- Helper functions ---
def regression_stats(xv, yv):
    xv = np.asarray(xv, dtype=float)
    yv = np.asarray(yv, dtype=float)

    if len(xv) < 3 or np.std(xv, ddof=1) == 0:
        return np.nan, np.nan, np.nan, np.nan

    slope, intercept = np.polyfit(xv, yv, 1)
    y_hat = intercept + slope * xv

    ss_res = np.sum((yv - y_hat) ** 2)
    ss_tot = np.sum((yv - np.mean(yv)) ** 2)

    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
    corr = np.corrcoef(xv, yv)[0, 1]

    return slope, intercept, r2, corr


def title_text(frame_date, n_points, slope, intercept, r2, corr):
    if np.isfinite(slope):
        stats = (
            f"Through {frame_date:%Y-%m-%d} · n={n_points:,} · "
            f"Regression: y = {slope:.2f}x + {intercept:.1%} · "
            f"R²={r2:.2f} · ρ={corr:.2f}"
        )
    else:
        stats = f"Through {frame_date:%Y-%m-%d} · n={n_points:,}"

    return (
        "Current Implied Vol vs Realized Vol 30 Trading Days Forward"
        f"<br><sup>{stats}</sup>"
    )


# --- Axis bounds ---
combined = np.concatenate([x, y])
axis_min = max(0, float(np.nanmin(combined) * 0.90))
axis_max = float(np.nanmax(combined) * 1.10)

line_x = np.linspace(axis_min, axis_max, 200)
identity_y = line_x

# --- Initial frame index ---
min_points_for_first_frame = 40
initial_i = min(min_points_for_first_frame, len(scatter_df) - 1)

slope0, intercept0, r20, corr0 = regression_stats(
    x[:initial_i + 1],
    y[:initial_i + 1]
)

reg_y0 = (
    intercept0 + slope0 * line_x
    if np.isfinite(slope0)
    else np.full_like(line_x, np.nan)
)

# --- Figure ---
fig = go.Figure()

# y = x line
fig.add_trace(go.Scatter(
    x=line_x,
    y=identity_y,
    mode="lines",
    line=dict(color=baseline_color, width=2, dash="dash"),
    opacity=0.85,
    name="y = x",
    hovertemplate="Equal IV / forward RV: %{x:.2%}<extra></extra>"
))

# Scatter points through time
fig.add_trace(go.Scatter(
    x=x[:initial_i + 1],
    y=y[:initial_i + 1],
    mode="markers",
    marker=dict(
        color=point_color,
        size=7,
        opacity=0.58,
        line=dict(color="rgba(255,255,255,0.25)", width=0.5)
    ),
    customdata=np.column_stack([
        dates.iloc[:initial_i + 1].dt.strftime("%Y-%m-%d"),
        x[:initial_i + 1],
        y[:initial_i + 1],
        y[:initial_i + 1] - x[:initial_i + 1]
    ]),
    name="Historical observations",
    hovertemplate=(
        "Date: %{customdata[0]}<br>"
        "Current IV: %{customdata[1]:.2%}<br>"
        "Realized Vol t+30: %{customdata[2]:.2%}<br>"
        "Forward RV − IV: %{customdata[3]:.2%}<extra></extra>"
    )
))

# Updating regression line
fig.add_trace(go.Scatter(
    x=line_x,
    y=reg_y0,
    mode="lines",
    line=dict(color=regression_color, width=4),
    name="Updating OLS fit",
    hovertemplate=(
        "OLS fit<br>"
        "Current IV: %{x:.2%}<br>"
        "Fitted t+30 RV: %{y:.2%}<extra></extra>"
    )
))

# Current latest point
fig.add_trace(go.Scatter(
    x=[x[initial_i]],
    y=[y[initial_i]],
    mode="markers",
    marker=dict(
        color=current_color,
        size=13,
        opacity=1.0,
        line=dict(color="rgba(255,255,255,0.95)", width=1.25)
    ),
    customdata=[[
        dates.iloc[initial_i].strftime("%Y-%m-%d"),
        x[initial_i],
        y[initial_i],
        y[initial_i] - x[initial_i]
    ]],
    name="Current frame point",
    hovertemplate=(
        "Frame date: %{customdata[0]}<br>"
        "Current IV: %{customdata[1]:.2%}<br>"
        "Realized Vol t+30: %{customdata[2]:.2%}<br>"
        "Forward RV − IV: %{customdata[3]:.2%}<extra></extra>"
    )
))

# Visual pricing gap labels
fig.add_annotation(
    x=axis_min + 0.78 * (axis_max - axis_min),
    y=axis_min + 0.90 * (axis_max - axis_min),
    text="t+30 RV > IV",
    showarrow=False,
    font=dict(color=current_color, size=13),
    opacity=0.8
)

fig.add_annotation(
    x=axis_min + 0.83 * (axis_max - axis_min),
    y=axis_min + 0.63 * (axis_max - axis_min),
    text="t+30 RV < IV",
    showarrow=False,
    font=dict(color=off_white, size=13),
    opacity=0.7
)

# --- Animation frames ---
frames = []
slider_steps = []

frame_stride = 10
frame_indices = list(range(initial_i, len(scatter_df), frame_stride))

if frame_indices[-1] != len(scatter_df) - 1:
    frame_indices.append(len(scatter_df) - 1)

for i in frame_indices:
    slope, intercept, r2, corr = regression_stats(x[:i + 1], y[:i + 1])

    reg_y = (
        intercept + slope * line_x
        if np.isfinite(slope)
        else np.full_like(line_x, np.nan)
    )

    frame_name = f"f{i}"

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=x[:i + 1],
                y=y[:i + 1],
                customdata=np.column_stack([
                    dates.iloc[:i + 1].dt.strftime("%Y-%m-%d"),
                    x[:i + 1],
                    y[:i + 1],
                    y[:i + 1] - x[:i + 1]
                ])
            ),
            go.Scatter(
                x=line_x,
                y=reg_y
            ),
            go.Scatter(
                x=[x[i]],
                y=[y[i]],
                customdata=[[
                    dates.iloc[i].strftime("%Y-%m-%d"),
                    x[i],
                    y[i],
                    y[i] - x[i]
                ]]
            )
        ],
        traces=[1, 2, 3],
        layout=go.Layout(
            title=dict(
                text=title_text(dates.iloc[i], i + 1, slope, intercept, r2, corr),
                x=0.5,
                font=dict(color=off_white)
            )
        ),
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": dates.iloc[i].strftime("%Y"),
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# --- Layout ---
fig.update_layout(
    title=dict(
        text=title_text(dates.iloc[initial_i], initial_i + 1, slope0, intercept0, r20, corr0),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=700,
    width=1100,
    margin=dict(t=105, b=150, r=50, l=85),
    legend=dict(
        x=0.98,
        y=0.04,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 28, "redraw": False},
                        "transition": {"duration": 18},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(
    axis_style,
    range=[axis_min, axis_max],
    title_text="Current Option-Implied Volatility",
    tickformat=".0%"
)

fig.update_yaxes(
    axis_style,
    range=[axis_min, axis_max],
    title_text="Realized Volatility 30 Trading Days Forward",
    tickformat=".0%",
    scaleanchor="x",
    scaleratio=1
)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Selling Insurance is a Profitable Business

The strategy is simnple in design, difficult to implement in practice

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Load data ---
path = "spx_option_implied_realized_vol_10y.csv"
df = pd.read_csv(path)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Use the supplied 21-day realized vol series and implied vol series.
# Drop the initial rows where the 21-day realized-vol window is not available.
df = df.dropna(subset=["realized_volatility", "option_implied_volatility"]).reset_index(drop=True)

realized_vol = df["realized_volatility"].to_numpy()
implied_vol = df["option_implied_volatility"].to_numpy()

# --- Styling ---
off_white = "#e0e0e0"
realized_color = "#00ff88"
implied_color = "#33aaff"
baseline_color = "#777777"

# --- Helper functions ---
def freedman_diaconis_bins(values, min_bins=25, max_bins=70):
    """Robust empirical bin count."""
    values = np.asarray(values)
    q25, q75 = np.percentile(values, [25, 75])
    iqr = q75 - q25
    n = len(values)
    if iqr <= 0:
        return min_bins
    width = 2 * iqr / np.cbrt(n)
    if width <= 0:
        return min_bins
    bins = int(np.ceil((values.max() - values.min()) / width))
    return int(np.clip(bins, min_bins, max_bins))

def gaussian_kde_manual(values, grid):
    """Simple Gaussian KDE using Silverman's bandwidth rule."""
    values = np.asarray(values, dtype=float)
    grid = np.asarray(grid, dtype=float)
    n = len(values)
    std = values.std(ddof=1)
    if std <= 0 or n <= 1:
        return np.zeros_like(grid)
    bandwidth = 1.06 * std * n ** (-1 / 5)
    z = (grid[:, None] - values[None, :]) / bandwidth
    density = np.exp(-0.5 * z**2).sum(axis=1) / (n * bandwidth * np.sqrt(2 * np.pi))
    return density

# --- Common empirical histogram setup ---
combined = np.concatenate([realized_vol, implied_vol])
x_min = max(0, combined.min() * 0.92)
x_max = combined.max() * 1.08

n_bins = freedman_diaconis_bins(combined)
bin_edges = np.linspace(x_min, x_max, n_bins + 1)
bin_width = bin_edges[1] - bin_edges[0]
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

rv_density, _ = np.histogram(realized_vol, bins=bin_edges, density=True)
iv_density, _ = np.histogram(implied_vol, bins=bin_edges, density=True)

x_grid = np.linspace(x_min, x_max, 500)
rv_kde = gaussian_kde_manual(realized_vol, x_grid)
iv_kde = gaussian_kde_manual(implied_vol, x_grid)

y_max = max(rv_density.max(), iv_density.max(), rv_kde.max(), iv_kde.max())
y_axis_top = y_max * 1.22
drop_start = y_axis_top * 1.10

# --- Summary stats ---
rv_mean = realized_vol.mean()
iv_mean = implied_vol.mean()
rv_median = np.median(realized_vol)
iv_median = np.median(implied_vol)
rv_std = realized_vol.std(ddof=1)
iv_std = implied_vol.std(ddof=1)

# --- Figure ---
fig = make_subplots(
    rows=1,
    cols=2,
    shared_yaxes=True,
    horizontal_spacing=0.08,
    subplot_titles=(
        f"21D Realized Vol · Mean {rv_mean:.1%}",
        f"Option-Implied Vol · Mean {iv_mean:.1%}"
    )
)

# Initial traces: histogram bars start above the visible plot area, then "fall" to base=0.
fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=rv_density,
        base=np.full_like(rv_density, drop_start),
        width=bin_width * 0.88,
        marker=dict(color=realized_color, opacity=0.72, line=dict(color="rgba(255,255,255,0.18)", width=0.5)),
        name="Realized Vol Histogram",
        customdata=np.column_stack([bin_edges[:-1], bin_edges[1:]]),
        hovertemplate="Bin: %{customdata[0]:.1%}–%{customdata[1]:.1%}<br>Density: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=x_grid,
        y=np.zeros_like(rv_kde),
        mode="lines",
        line=dict(color=realized_color, width=3),
        name="Realized Vol KDE",
        hovertemplate="Vol: %{x:.2%}<br>KDE density: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Bar(
        x=bin_centers,
        y=iv_density,
        base=np.full_like(iv_density, drop_start),
        width=bin_width * 0.88,
        marker=dict(color=implied_color, opacity=0.72, line=dict(color="rgba(255,255,255,0.18)", width=0.5)),
        name="Implied Vol Histogram",
        customdata=np.column_stack([bin_edges[:-1], bin_edges[1:]]),
        hovertemplate="Bin: %{customdata[0]:.1%}–%{customdata[1]:.1%}<br>Density: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=x_grid,
        y=np.zeros_like(iv_kde),
        mode="lines",
        line=dict(color=implied_color, width=3),
        name="Implied Vol KDE",
        hovertemplate="Vol: %{x:.2%}<br>KDE density: %{y:.2f}<extra></extra>"
    ),
    row=1,
    col=2
)

# Mean reference lines as static shapes.
for col, mean_val, color in [(1, rv_mean, realized_color), (2, iv_mean, implied_color)]:
    fig.add_vline(
        x=mean_val,
        line=dict(color=color, width=1, dash="dash"),
        opacity=0.75,
        row=1,
        col=col
    )

# --- Animation frames ---
frames = []
slider_steps = []
n_frames = 56

for k in range(n_frames):
    t = k / (n_frames - 1)
    # Smooth falling / easing motion
    p = 1 - (1 - t) ** 3
    
    # Slightly delay the KDE so the bars land first, then the fitted curve settles on top.
    kde_p = np.clip((p - 0.15) / 0.85, 0, 1)
    kde_p = 1 - (1 - kde_p) ** 3
    
    current_base = np.full_like(rv_density, drop_start * (1 - p))
    frame_name = f"f{k}"
    
    frames.append(go.Frame(
        data=[
            go.Bar(x=bin_centers, y=rv_density, base=current_base),
            go.Scatter(x=x_grid, y=rv_kde * kde_p),
            go.Bar(x=bin_centers, y=iv_density, base=current_base),
            go.Scatter(x=x_grid, y=iv_kde * kde_p),
        ],
        name=frame_name
    ))
    
    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": f"{int(round(p * 100)):d}%",
        "method": "animate"
    })

fig.frames = frames

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# --- Layout ---
fig.update_layout(
    title=dict(
        text=(
            "Empirical Distributions of SPX Realized and Implied Volatility"
            f"<br><sup>Density-normalized histograms with Gaussian KDE overlays · "
            f"RV median: {rv_median:.1%} · IV median: {iv_median:.1%} · "
            f"RV σ: {rv_std:.1%} · IV σ: {iv_std:.1%}</sup>"
        ),
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=650,
    width=1100,
    margin=dict(t=110, b=150, r=50, l=75),
    legend=dict(
        x=0.98,
        y=0.96,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(30,30,30,0.8)",
        bordercolor="rgba(255,255,255,0.25)",
        borderwidth=1,
        font=dict(color=off_white)
    ),
    bargap=0.05,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 28, "redraw": False},
                        "transition": {"duration": 18},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Formation: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(axis_style, range=[x_min, x_max], title_text="Annualized Volatility", tickformat=".0%", row=1, col=1)
fig.update_xaxes(axis_style, range=[x_min, x_max], title_text="Annualized Volatility", tickformat=".0%", row=1, col=2)
fig.update_yaxes(axis_style, range=[0, y_axis_top], title_text="Density", row=1, col=1)
fig.update_yaxes(axis_style, range=[0, y_axis_top], row=1, col=2)

# Style subplot titles
for annotation in fig.layout.annotations:
    annotation.font = dict(color=off_white, size=15)

fig.show()

###### ______________________________________________________________________________________________________________________________________

##### Buying Insurance is Also a Profitable Business

How can they both be true at the same time?  *Timing*.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =============================================================================
# Stylized insurance / VRP simulation
# =============================================================================

np.random.seed(7)

n_steps = 252
dates = pd.date_range("2025-01-01", periods=n_steps + 1, freq="B")
start_value = 100.0

# Market path with one severe drawdown event
rets = np.random.normal(0.00035, 0.0055, n_steps)

crash_start = 92
crash_end = 126

rets[crash_start:crash_end] += -0.018
rets[104] += -0.055
rets[113] += -0.040

# Partial recovery
rets[crash_end:185] += 0.0045

market = start_value * np.cumprod(1 + rets)
market = np.insert(market, 0, start_value)

# =============================================================================
# Insurance economics
# =============================================================================

annual_premium_rate = 0.055
daily_premium = start_value * annual_premium_rate / 252
cumulative_premium = np.arange(n_steps + 1) * daily_premium

protective_floor = 85.0

# Insurance payout kicks in when the market breaches the floor.
cumulative_payout = np.maximum(0, protective_floor - market)
cumulative_payout = np.maximum.accumulate(cumulative_payout)

# Seller: earns premium, pays claim.
insurance_seller = start_value + cumulative_premium - cumulative_payout

# Buyer: owns market, pays premium, receives claim.
insured_equity = market - cumulative_premium + cumulative_payout

# =============================================================================
# Drawdowns
# =============================================================================

def drawdown(series):
    series = np.asarray(series, dtype=float)
    return series / np.maximum.accumulate(series) - 1

market_dd = drawdown(market)
insured_dd = drawdown(insured_equity)
seller_dd = drawdown(insurance_seller)

max_market_dd = market_dd.min()
max_insured_dd = insured_dd.min()
max_seller_dd = seller_dd.min()

# =============================================================================
# Styling
# =============================================================================

off_white = "#e0e0e0"
seller_color = "#ff4fd8"
market_color = "#777777"
insured_color = "#00ff88"
baseline_color = "#777777"

axis_style = dict(
    showgrid=True,
    gridcolor="rgba(255,255,255,0.1)",
    tickfont=dict(color=off_white),
    linecolor=off_white,
    zeroline=False,
    title_font=dict(color=off_white)
)

# =============================================================================
# Figure
# =============================================================================

fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.08,
    subplot_titles=(
        f"Selling Insurance · MDD {max_seller_dd:.1%}",
        f"Buying Insurance · Market MDD {max_market_dd:.1%} vs Insured MDD {max_insured_dd:.1%}"
    )
)

# --- Left chart: insurance seller ---

fig.add_trace(
    go.Scatter(
        x=[dates[0], dates[-1]],
        y=[start_value, start_value],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.45,
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=1
)

fig.add_trace(
    go.Scatter(
        x=[dates[0]],
        y=[insurance_seller[0]],
        mode="lines",
        line=dict(color=seller_color, width=4),
        showlegend=False,
        customdata=np.column_stack([
            seller_dd[:1],
            cumulative_premium[:1],
            cumulative_payout[:1]
        ]),
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Seller value: %{y:.2f}<br>"
            "Drawdown: %{customdata[0]:.1%}<br>"
            "Premium collected: %{customdata[1]:.2f}<br>"
            "Payout owed: %{customdata[2]:.2f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=1
)

# --- Right chart: market vs insured equity ---

fig.add_trace(
    go.Scatter(
        x=[dates[0], dates[-1]],
        y=[start_value, start_value],
        mode="lines",
        line=dict(color=baseline_color, width=1, dash="dash"),
        opacity=0.45,
        hoverinfo="skip",
        showlegend=False
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[dates[0]],
        y=[market[0]],
        mode="lines",
        line=dict(color=market_color, width=3),
        opacity=0.85,
        showlegend=False,
        customdata=np.column_stack([market_dd[:1]]),
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Market value: %{y:.2f}<br>"
            "Drawdown: %{customdata[0]:.1%}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

fig.add_trace(
    go.Scatter(
        x=[dates[0]],
        y=[insured_equity[0]],
        mode="lines",
        line=dict(color=insured_color, width=4),
        showlegend=False,
        customdata=np.column_stack([
            insured_dd[:1],
            cumulative_premium[:1],
            cumulative_payout[:1]
        ]),
        hovertemplate=(
            "Date: %{x|%Y-%m-%d}<br>"
            "Insured value: %{y:.2f}<br>"
            "Drawdown: %{customdata[0]:.1%}<br>"
            "Premium paid: %{customdata[1]:.2f}<br>"
            "Payout received: %{customdata[2]:.2f}"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)

# Crash-window shading only
for col in [1, 2]:
    fig.add_vrect(
        x0=dates[crash_start],
        x1=dates[crash_end],
        fillcolor="rgba(255,62,62,0.10)",
        line_width=0,
        layer="below",
        row=1,
        col=col
    )

# =============================================================================
# Animation
# =============================================================================

frames = []
slider_steps = []

frame_stride = 4
frame_indices = list(range(0, n_steps + 1, frame_stride))

if frame_indices[-1] != n_steps:
    frame_indices.append(n_steps)

for i in frame_indices:
    frame_name = f"f{i}"

    frames.append(go.Frame(
        data=[
            go.Scatter(
                x=dates[:i + 1],
                y=insurance_seller[:i + 1],
                customdata=np.column_stack([
                    seller_dd[:i + 1],
                    cumulative_premium[:i + 1],
                    cumulative_payout[:i + 1]
                ])
            ),
            go.Scatter(
                x=dates[:i + 1],
                y=market[:i + 1],
                customdata=np.column_stack([
                    market_dd[:i + 1]
                ])
            ),
            go.Scatter(
                x=dates[:i + 1],
                y=insured_equity[:i + 1],
                customdata=np.column_stack([
                    insured_dd[:i + 1],
                    cumulative_premium[:i + 1],
                    cumulative_payout[:i + 1]
                ])
            )
        ],
        traces=[1, 3, 4],
        name=frame_name
    ))

    slider_steps.append({
        "args": [
            [frame_name],
            {
                "frame": {"duration": 0, "redraw": False},
                "mode": "immediate",
                "fromcurrent": True
            }
        ],
        "label": dates[i].strftime("%b"),
        "method": "animate"
    })

fig.frames = frames

# =============================================================================
# Layout
# =============================================================================

y_min = min(
    insurance_seller.min(),
    market.min(),
    insured_equity.min()
)

y_max = max(
    insurance_seller.max(),
    market.max(),
    insured_equity.max()
)

y_pad = (y_max - y_min) * 0.12

fig.update_layout(
    title=dict(
        text="Volatility Risk Premium as Insurance Economics",
        x=0.5,
        font=dict(color=off_white)
    ),
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    height=680,
    width=1250,
    margin=dict(t=100, b=145, r=55, l=75),
    showlegend=False,
    hovermode="closest",
    updatemenus=[{
        "type": "buttons",
        "buttons": [
            {
                "label": "▶ Play",
                "method": "animate",
                "args": [
                    None,
                    {
                        "frame": {"duration": 35, "redraw": False},
                        "transition": {"duration": 0},
                        "fromcurrent": True
                    }
                ]
            },
            {
                "label": "⏸ Pause",
                "method": "animate",
                "args": [
                    [None],
                    {
                        "frame": {"duration": 0, "redraw": False},
                        "mode": "immediate",
                        "fromcurrent": True
                    }
                ]
            }
        ],
        "direction": "left",
        "pad": {"r": 10, "t": 87},
        "showactive": False,
        "x": 0.1,
        "xanchor": "right",
        "y": 0,
        "yanchor": "top"
    }],
    sliders=[{
        "active": 0,
        "yanchor": "top",
        "xanchor": "left",
        "currentvalue": {
            "font": {"size": 14, "color": off_white},
            "prefix": "Through: ",
            "visible": True,
            "xanchor": "right"
        },
        "transition": {"duration": 0},
        "pad": {"b": 10, "t": 50},
        "len": 0.85,
        "x": 0.15,
        "y": 0,
        "steps": slider_steps
    }]
)

fig.update_xaxes(
    axis_style,
    range=[dates[0], dates[-1]],
    title_text="Date",
    row=1,
    col=1
)

fig.update_xaxes(
    axis_style,
    range=[dates[0], dates[-1]],
    title_text="Date",
    row=1,
    col=2
)

fig.update_yaxes(
    axis_style,
    range=[y_min - y_pad, y_max + y_pad],
    title_text="Strategy Value",
    row=1,
    col=1
)

fig.update_yaxes(
    axis_style,
    range=[y_min - y_pad, y_max + y_pad],
    title_text="Portfolio Value",
    row=1,
    col=2
)

# Style subplot titles
for annotation in fig.layout.annotations:
    annotation.font = dict(color=off_white, size=15)

fig.show()

---

#### 💭 Closing Thoughts and Future Topics

 **📑 TL;DW Executive Summary**

 - Implied volatility captures the market’s expectations for future price swings, while realized volatility reflects what actually happens. Both are essential for understanding risk and reward in options trading.

 - Our analysis showed that implied and realized volatility often diverge, creating opportunities for traders who can identify and exploit this gap. By visualizing historical data, we made these dynamics clear.

 - Systematic, rules-based volatility strategies—rather than relying on market timing—can help build more resilient portfolios, especially during periods of heightened market uncertainty.

 
**Future Topics**

Technical Videos and Other Discussions

 - Fama-French / Carhart and Factor Modeling in General
 - Hawkes Processes
 - Merton Jump Diffusion Model (and Characteristic Function Pricing, Carr-Madan 1999)
 - Market-Making Models and Simulation (Stoikov-Avellaneda)
 - My First Year as a Quant
 - Why Hedge Funds are Actually Secretive
 - Non-Markovian Models (fractional Brownian motion, Volterra Process)
 - Top 3 Uses of Linear Algebra for Quant Finance
 - Girsanov's Change of Measure
 - Rough Path Theory, Applications of Path Signatures
 - Sig-Vol Model, Calibration, and Pricing
 - Trading with Alternative Data Sources
 - Pairs Trading and Statistical Arbitrage
 - Data Cleaning & Outlier Handling in Financial Time Series
 - Practical Issues in Multi-Asset Portfolio Backtesting
 - Risk Premia Harvesting: Equity, FX, Rates

[Ideas for Interactive Brokers Apps and Tutorials](https://www.interactivebrokers.com/mkt/?src=quantguildY&url=%2Fen%2Fwhyib%2Foverview.php)

- How Interactive Broker's API Works (EWrapper/EClient)
- How to Backtest a Trading Strategy with Interactive Brokers
- Algorithmic Volatility Trading System

---

####  $\text{Copyright © 2026 Quant Guild} \quad \quad \quad \quad \text{Author: Roman Paolucci}$